# Supplementary Figure — detailed K=46 edge-trim sensitivity

This figure expands the compact robustness panel in the main text.

It uses the CSV summaries exported by the K=46 edge-trim analysis:
- baseline vs trimmed top-5 accuracy;
- above-chance fraction preserved;
- top-5 set overlap;
- selection-frequency stability for the five locked interpretation motifs.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EDGE_DIR = Path("msaa_edge_trim_sensitivity_K46_top5")

ACC_CSV = EDGE_DIR / "edge_trim_accuracy_summary.csv"
JACCARD_CSV = EDGE_DIR / "edge_trim_top5_jaccard_summary.csv"
FREQ_CSV = EDGE_DIR / "edge_trim_selection_frequency_comparison.csv"
ARCH_CSV = EDGE_DIR / "edge_trim_locked_archetype_single_component_summary.csv"

CONDITIONS = ["intact", "word", "rest"]
COLORS = {
    "intact": "#7B3294",
    "word": "#2CA25F",
    "rest": "#222222",
}
LOCKED = [4, 5, 6, 12, 8]

OUTPUT_DIR = Path("outputs/paper_supplement")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_PNG = OUTPUT_DIR / "Supplement_edge_trim_diagnostics.png"
FIG_PDF = OUTPUT_DIR / "Supplement_edge_trim_diagnostics.pdf"

for p in [ACC_CSV, JACCARD_CSV, FREQ_CSV, ARCH_CSV]:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing {p}. Run revision_08_K46_edge_trim_heldout_top5_sensitivity first."
        )

acc = pd.read_csv(ACC_CSV)
jacc = pd.read_csv(JACCARD_CSV)
freq = pd.read_csv(FREQ_CSV)
arch = pd.read_csv(ARCH_CSV)

In [ ]:
fig, axes = plt.subplots(
    2,
    2,
    figsize=(11.5, 8.3),
    gridspec_kw={"hspace": 0.35, "wspace": 0.28},
)

axA, axB, axC, axD = axes.ravel()

# A — raw accuracy
plot_acc = acc.set_index("condition").reindex(CONDITIONS)
x = np.arange(len(CONDITIONS))
offset = 0.10

axA.errorbar(
    x - offset,
    plot_acc["baseline_accuracy"],
    yerr=plot_acc["baseline_accuracy_sem"],
    marker="o",
    linewidth=1.8,
    capsize=3,
    label="Untrimmed",
)

axA.errorbar(
    x + offset,
    plot_acc["trim_accuracy"],
    yerr=plot_acc["trim_accuracy_sem"],
    marker="s",
    linewidth=1.8,
    capsize=3,
    label="Trim first/last 10 TRs",
)

axA.set_xticks(x)
axA.set_xticklabels(["Intact", "Word", "Rest"])
axA.set_ylabel("Held-out top-5 accuracy")
axA.set_title("Accuracy after edge exclusion", fontweight="bold")
axA.legend(frameon=False, fontsize=9)

# B — above-chance fraction preserved
axB.bar(
    x,
    plot_acc["fraction_preserved"],
    color=[COLORS[c] for c in CONDITIONS],
)
axB.axhline(1.0, color="0.5", linestyle=":", linewidth=1)
axB.set_xticks(x)
axB.set_xticklabels(["Intact", "Word", "Rest"])
axB.set_ylabel("Fraction of above-chance signal preserved")
axB.set_title("Above-chance information preserved", fontweight="bold")

# C — top-5 set overlap
pj = jacc.set_index("condition").reindex(CONDITIONS)
axC.bar(
    x,
    pj["mean_jaccard"],
    color=[COLORS[c] for c in CONDITIONS],
)
axC.set_ylim(0, 1)
axC.set_xticks(x)
axC.set_xticklabels(["Intact", "Word", "Rest"])
axC.set_ylabel("Mean Jaccard overlap")
axC.set_title("Selected-set stability", fontweight="bold")

# D — locked motifs: trimmed - baseline selection frequency
lf = freq[freq["archetype"].isin(LOCKED)].copy()
lf["delta_frequency"] = (
    lf["trim_frequency"]
    - lf["baseline_frequency"]
)

heat = (
    lf.pivot_table(
        index="archetype",
        columns="condition",
        values="delta_frequency",
        aggfunc="mean",
    )
    .reindex(index=LOCKED, columns=CONDITIONS)
)

arr = heat.to_numpy(dtype=float)
v = np.nanmax(np.abs(arr))
if not np.isfinite(v) or v == 0:
    v = 1.0

im = axD.imshow(
    arr,
    aspect="auto",
    cmap="coolwarm",
    vmin=-v,
    vmax=v,
)

axD.set_xticks(np.arange(len(CONDITIONS)))
axD.set_xticklabels(["Intact", "Word", "Rest"])
axD.set_yticks(np.arange(len(LOCKED)))
axD.set_yticklabels([f"A{k}" for k in LOCKED])
axD.set_title("Change in top-5 selection frequency", fontweight="bold")

for r in range(arr.shape[0]):
    for c in range(arr.shape[1]):
        if np.isfinite(arr[r, c]):
            axD.text(
                c, r, f"{arr[r, c]:+.2f}",
                ha="center", va="center",
                fontsize=9,
            )

cb = fig.colorbar(im, ax=axD, fraction=0.046, pad=0.04)
cb.set_label("Trimmed − untrimmed")

for ax, letter in zip(axes.ravel(), list("ABCD")):
    ax.text(
        -0.10, 1.04, letter,
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    "Detailed edge-trim sensitivity of the K=46 held-out top-5 result",
    fontsize=16,
    fontweight="bold",
)

fig.savefig(FIG_PNG, dpi=300, bbox_inches="tight")
fig.savefig(FIG_PDF, bbox_inches="tight")
plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)